# Week 6: Integrated Data Science Project
## Breast cancer diagnosis classification (educational benchmark)
This notebook covers data loading, cleaning, EDA, preprocessing, model training, evaluation, visualization, and limitations. **It is not a clinical diagnostic tool.**

In [ ]:
import pandas as pd\nimport matplotlib.pyplot as plt\nfrom sklearn.datasets import load_breast_cancer\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay

## 1. Load and inspect the dataset

In [ ]:
ds = load_breast_cancer(as_frame=True)\ndata = ds.frame.copy()\ndata['diagnosis'] = data['target'].map({0: 'malignant', 1: 'benign'})\ndata = data.drop(columns=['target'])\nprint('Shape:', data.shape)\ndisplay(data.head())\nprint(data.isna().sum().sum(), 'missing cells')\nprint('Duplicate rows:', data.duplicated().sum())\ndisplay(data['diagnosis'].value_counts())

## 2. Exploratory data analysis

In [ ]:
data['diagnosis'].value_counts().reindex(['benign','malignant']).plot(kind='bar', title='Diagnosis class distribution')\nplt.xlabel('Diagnosis'); plt.ylabel('Samples'); plt.xticks(rotation=0); plt.show()\nfor label in ['benign','malignant']:\n    plt.hist(data.loc[data.diagnosis == label, 'mean radius'], bins=20, alpha=.55, label=label)\nplt.title('Mean radius by diagnosis'); plt.xlabel('Mean radius'); plt.ylabel('Frequency'); plt.legend(); plt.show()

## 3. Prepare features and split data
The positive class is malignant. A stratified 80/20 split preserves the class proportion. Imputation and scaling are fitted inside a pipeline on training data only to avoid leakage.

In [ ]:
X = data.drop(columns=['diagnosis'])\ny = (data['diagnosis'] == 'malignant').astype(int)\nX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)\nmodel = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('classifier', LogisticRegression(max_iter=5000, random_state=42))])\nmodel.fit(X_train, y_train)\npred = model.predict(X_test)\nprob = model.predict_proba(X_test)[:, 1]

## 4. Evaluate the model
Accuracy alone is insufficient for medical classification. Recall for the malignant class is especially important, but false positives and false negatives both matter.

In [ ]:
metrics = {'accuracy': accuracy_score(y_test,pred), 'precision_malignant': precision_score(y_test,pred), 'recall_malignant': recall_score(y_test,pred), 'f1_malignant': f1_score(y_test,pred), 'roc_auc': roc_auc_score(y_test,prob)}\npd.Series(metrics).round(4)

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=['benign','malignant'], values_format='d')\nplt.title('Confusion matrix'); plt.show()\nRocCurveDisplay.from_predictions(y_test, prob)\nplt.title('ROC curve'); plt.show()

## 5. Interpretation, limitations, and next steps
The model is a reproducible baseline, not a medical device. The dataset is a curated benchmark and may not represent real-world clinical populations. Do not use predictions for diagnosis or treatment. Further work should include cross-validation, calibration, external validation, subgroup performance analysis, uncertainty estimates, and review by qualified clinicians.